# Modelo predictivo avanzado – Precio de Airbnb en Nueva York (2019)

**Objetivo:** predecir el precio por noche (`price`) de un alojamiento de Airbnb en Nueva York. Es un problema de **regresión**, por lo que todas las medidas de selección buscan **minimizar el error**.

Este notebook aplica el punto 3 del proyecto sobre las **34 características** construidas en la minería de características (notebook `04_mineria_caracteristicas`), que están definidas en `features.py`, y guarda el modelo que se despliega en la aplicación Streamlit.

**Contenido**
1. Librerías y carga de datos
2. Ingeniería de características (a)
3. División 70 % / 30 %
4. Balanceo (b)
5. Modelos: 4 métodos de aprendizaje supervisado y 3 métodos de ensamble (d, e)
6. Validación cruzada con el 70 % y medidas de calidad (c, f)
7. Selección de los 3 mejores modelos con ANOVA y Tukey (g)
8. Hiperparametrización: GridSearch y optimización bayesiana (h)
9. Evaluación con el 30 % de prueba y comparación con el modelo base
10. Pipeline final para despliegue (i)

El despliegue con Streamlit (j) está en el notebook **Despliegue_Airbnb_NYC**.

*Para ejecutarlo en Colab: subir `features.py` y `AB_NYC_2019.csv` al panel de archivos. En el repositorio se ejecuta desde la carpeta `notebooks/`. Tiempo aproximado con 2 CPU: 60–80 minutos (la sección 8 es la más larga).*

# 1. Librerías y carga de datos

In [ ]:
import os
import sys
import json
import time
import subprocess
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import optuna
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'], check=True)
    import optuna

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error, median_absolute_error,
                             mean_absolute_percentage_error, max_error)

# Métodos de aprendizaje supervisado
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

# Métodos de ensamble
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Pruebas estadísticas
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Carpeta raíz del proyecto: donde está features.py (la raíz del repositorio o la carpeta de Colab)
RAIZ = next(d for d in ['.', '..'] if os.path.exists(os.path.join(d, 'features.py')))
sys.path.insert(0, RAIZ)
from features import (preparar_datos, crear_variables, construir_pipeline,
                      COLUMNAS_NUMERICAS, COLUMNAS_CATEGORICAS_OHE, COLUMNAS_CATEGORICAS_TARGET)

def ruta(*partes):
    # Busca el archivo en la estructura del repositorio y, si no existe, en la carpeta actual (Colab)
    completa = os.path.join(RAIZ, *partes)
    return completa if os.path.exists(os.path.join(RAIZ, partes[0])) else partes[-1]

print('Raíz del proyecto:', os.path.abspath(RAIZ))

In [ ]:
datos = pd.read_csv(ruta('data', 'AB_NYC_2019.csv'))
print('Datos originales:', datos.shape)

# Mismas reglas de calidad del notebook de preparación (solo se eliminan errores)
datos = datos.drop_duplicates(subset='id', keep='last')
datos = datos[datos['price'] > 0]
datos = datos[(datos['minimum_nights'] >= 1) & (datos['minimum_nights'] <= 365)]
print('Datos después de las reglas de calidad:', datos.shape)

# 2. Ingeniería de características (a)

En la preparación se descartaron `name` (título) y `neighbourhood` (micro-barrio). La minería de características (notebook 04) mostró que sí aportan información sobre el precio, por lo que se recuperan y se transforman. Todas las operaciones están en `features.py` y se ejecutan **dentro del pipeline**, así que en cada pliegue de la validación cruzada se ajustan solo con los datos de entrenamiento.

| Operación | Variables creadas | Justificación |
|---|---|---|
| **Construcción geográfica**: distancia *haversine* a 6 puntos de interés y al aeropuerto más cercano | `dist_times_square_km`, `dist_central_park_km`, `dist_fidi_km`, `dist_brooklyn_bridge_km`, `dist_jfk_km`, `dist_lga_km`, `dist_min_aeropuerto_km` | La latitud y la longitud por separado tienen una relación débil y no lineal con el precio; las distancias a lugares de referencia la resumen. |
| **Construcción de texto** a partir del título | `feat_luxury_count`, `feat_has_luxury`, `feat_budget_count`, `feat_has_budget`, `feat_transit_count`, `feat_spacious_count`, `name_length`, `name_word_count` | Palabras como *luxury*, *penthouse* o *cozy* indican el segmento del alojamiento. |
| **Ratios de negocio** | `availability_ratio`, `is_full_year_available`, `is_host_multi_listing` | Resumen la disponibilidad y si el anfitrión es profesional. |
| **Transformación logarítmica** `log(1 + x)` | `minimum_nights`, `number_of_reviews`, `reviews_per_month`, `calculated_host_listings_count` | Son conteos muy sesgados a la derecha. |
| **Transformación logarítmica del objetivo** | `price` | El precio es muy asimétrico (mediana ≈ 106 USD, máximo 10 000 USD); el modelo aprende `log(1 + price)` y devuelve dólares. |
| **Target Encoding** con validación cruzada interna (5 pliegues) | `neighbourhood` (221 barrios) | Convierte cada barrio en un número sin crear 221 columnas y sin fuga de información. |
| **One-hot** | `neighbourhood_group`, `room_type` | Categorías sin orden. |
| **Estandarización** | Variables numéricas | Necesaria para Ridge, KNN y la red neuronal. |

In [ ]:
# Comprobación de las variables creadas sobre una muestra del conjunto
muestra = crear_variables(preparar_datos(datos.drop(columns='price')))
nuevas = ['dist_times_square_km', 'dist_fidi_km', 'dist_min_aeropuerto_km', 'feat_luxury_count',
          'feat_budget_count', 'feat_spacious_count', 'name_length', 'availability_ratio', 'is_host_multi_listing']
print('Columnas que recibe el preprocesamiento:',
      len(COLUMNAS_NUMERICAS) + len(COLUMNAS_CATEGORICAS_OHE) + len(COLUMNAS_CATEGORICAS_TARGET))
print('\nCorrelación de Spearman de algunas variables creadas con el precio:')
print(muestra[nuevas].corrwith(datos['price'], method='spearman').round(3).sort_values())

# 3. División de los datos: 70 % entrenamiento y 30 % prueba

In [ ]:
X = datos.drop(columns='price')
y = datos['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
print('Entrenamiento (70 %):', X_train.shape)
print('Prueba (30 %):', X_test.shape)

El 30 % de prueba no se usa en ningún paso de entrenamiento ni de selección; se reserva para la evaluación final.

# 4. Balanceo (b)

El balanceo (submuestreo, sobremuestreo, SMOTE) se aplica en **clasificación**, cuando una clase tiene muchos menos registros que otra. Aquí la variable objetivo, `price`, es **numérica continua**, así que no hay clases que balancear y **el balanceo no es necesario**. El problema equivalente en regresión, la asimetría del precio, se trata con la transformación logarítmica del objetivo:

In [ ]:
print('Asimetría de price:', round(y_train.skew(), 2))
print('Asimetría de log(1 + price):', round(np.log1p(y_train).skew(), 2))

# 5. Modelos (d, e)

**4 métodos de aprendizaje supervisado:** regresión Ridge, K vecinos más cercanos (KNN), árbol de decisión y red neuronal (perceptrón multicapa, MLP).

**3 métodos de ensamble:** Random Forest (*bagging*), Gradient Boosting y XGBoost (*boosting*).

Todos usan el mismo pipeline de `features.py`: preparación → ingeniería de características → preprocesamiento → modelo, con el objetivo en escala logarítmica.

In [ ]:
modelos = {
    'Ridge': Ridge(alpha=1.0),
    'KNN': KNeighborsRegressor(n_neighbors=15),
    'Árbol de decisión': DecisionTreeRegressor(max_depth=10, min_samples_leaf=20, random_state=42),
    'Red neuronal (MLP)': MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=300, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, n_jobs=-1, random_state=42),
}

# 6. Validación cruzada con el 70 % y medidas de calidad (c, f)

Validación cruzada de **10 pliegues** (*K-Fold*) sobre el 70 % de entrenamiento. En cada pliegue se calculan 8 medidas. Siete son medidas de **error (menor es mejor)** y el R² es complementario:

| Medida | Qué mide | Mejor si |
|---|---|---|
| **MAE** | Error absoluto medio, en dólares. **Criterio principal de selección.** | Más bajo |
| **MedAE** | Mediana del error absoluto: el error de un alojamiento típico. | Más bajo |
| **RMSE** | Raíz del error cuadrático medio, en dólares. Penaliza los errores grandes. | Más bajo |
| **MSE** | Error cuadrático medio, en dólares². | Más bajo |
| **MAPE** | Error porcentual absoluto medio (0.30 = 30 %). | Más bajo |
| **Error máximo** | Mayor error en un solo alojamiento. | Más bajo |
| **R² (escala log)** | Variabilidad de `log(price)` explicada; es la escala en la que aprende el modelo. | Más alto |
| **R² (USD)** | Variabilidad del precio en dólares explicada; muy afectada por los precios extremos. | Más alto |

In [ ]:
def calcular_medidas(y_real, y_pred):
    return {
        'MAE': mean_absolute_error(y_real, y_pred),
        'MedAE': median_absolute_error(y_real, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_real, y_pred)),
        'MSE': mean_squared_error(y_real, y_pred),
        'MAPE': mean_absolute_percentage_error(y_real, y_pred),
        'Error_max': max_error(y_real, y_pred),
        'R2_log': r2_score(np.log1p(y_real), np.log1p(y_pred)),
        'R2_USD': r2_score(y_real, y_pred),
    }


kfold = KFold(n_splits=10, shuffle=True, random_state=42)
resultados = []
tiempos = {}

for nombre, modelo in modelos.items():
    inicio = time.time()
    for pliegue, (idx_ent, idx_val) in enumerate(kfold.split(X_train), start=1):
        pipe = construir_pipeline(modelo)
        pipe.fit(X_train.iloc[idx_ent], y_train.iloc[idx_ent])
        medidas = calcular_medidas(y_train.iloc[idx_val], pipe.predict(X_train.iloc[idx_val]))
        medidas.update({'modelo': nombre, 'pliegue': pliegue})
        resultados.append(medidas)
    tiempos[nombre] = time.time() - inicio
    print(f'{nombre:<20} 10 pliegues en {tiempos[nombre]:.1f} s')

resultados_cv = pd.DataFrame(resultados)

In [ ]:
# Promedio de las medidas en los 10 pliegues, ordenado por MAE (menor es mejor)
resumen = resultados_cv.drop(columns='pliegue').groupby('modelo').mean().sort_values('MAE')
resumen['Tiempo_s'] = pd.Series(tiempos)
resumen.round(3)

In [ ]:
medidas = ['MAE', 'MedAE', 'RMSE', 'MSE', 'MAPE', 'Error_max', 'R2_log', 'R2_USD']
fig, ejes = plt.subplots(2, 4, figsize=(22, 9))
for eje, medida in zip(ejes.ravel(), medidas):
    sns.boxplot(data=resultados_cv, x='modelo', y=medida, order=resumen.index, color='skyblue', ax=eje)
    eje.set_title(medida)
    eje.set_xlabel('')
    eje.tick_params(axis='x', rotation=45)
plt.suptitle('Medidas de calidad en los 10 pliegues de la validación cruzada', fontsize=14)
plt.tight_layout()
plt.show()

**Interpretación de las medidas de calidad (validación cruzada con el 70 %)**

* **MAE (55.0 – 60.7 USD):** con XGBoost, la predicción se aleja en promedio 55.0 USD del precio real; Random Forest queda muy cerca (55.3 USD) y KNN es el peor (60.7 USD).
* **MedAE (23.4 – 27.6 USD):** para la mitad de los alojamientos, el error de XGBoost es menor que 23.4 USD. Es la medida que mejor describe el error en un alojamiento típico.
* **RMSE (203 – 210 USD) y MSE (43 636 – 46 299 USD²):** el RMSE es unas 3.7 veces el MAE. Como elevan los errores al cuadrado, están dominados por unos pocos alojamientos de miles de dólares; el MSE, en dólares al cuadrado, no se interpreta directamente.
* **MAPE (0.313 – 0.369):** con XGBoost, el error equivale en promedio al 31 % del precio.
* **Error máximo (6 314 – 6 407 USD, promedio de los 10 pliegues):** casi igual en los 7 modelos; en cada pliegue el peor error es un alojamiento de miles de dólares que ningún modelo puede anticipar con estas variables.
* **R² en escala logarítmica (0.51 – 0.62):** XGBoost explica el 62 % de la variabilidad del logaritmo del precio. El dataset no incluye variables decisivas como el tamaño o el número de habitaciones.
* **R² en dólares (0.14 – 0.20):** mucho más bajo por los mismos precios extremos que inflan el RMSE.
* **Tiempo de aprendizaje:** XGBoost es el más preciso y de los más rápidos (9 s para 10 pliegues), frente a 181 s de Random Forest y 129 s de Gradient Boosting.

**Comparación:** XGBoost es el mejor en las 8 medidas y Random Forest el segundo en casi todas. Les siguen Gradient Boosting y la red neuronal. Frente a la primera versión del proyecto (9 variables), las 34 características bajan el MAE de validación cruzada de XGBoost de 55.9 a 55.0 USD; KNN empeora (de 58.6 a 60.7 USD) porque con muchas variables las distancias entre alojamientos pierden significado. En los diagramas de caja, la variación entre pliegues es comparable a las diferencias entre modelos, por lo que hace falta una prueba estadística.

# 7. Selección de los 3 mejores modelos: ANOVA y Tukey (g)

Se compara el **MAE de los 10 pliegues** de los 7 modelos, porque el MAE está en dólares, es la medida de error que se busca minimizar y no está dominado por los pocos precios extremos (como el RMSE o el R² en dólares).

* **ANOVA de un factor**: H₀: los 7 modelos tienen el mismo MAE medio; H₁: al menos uno es distinto.
* **Tukey (HSD)**: si el ANOVA es significativo, compara los modelos por pares controlando el error por comparaciones múltiples.

Nivel de significancia: α = 0.05.

In [ ]:
grupos_mae = [resultados_cv.loc[resultados_cv['modelo'] == nombre, 'MAE'] for nombre in modelos]
F, p_valor = f_oneway(*grupos_mae)
print(f'ANOVA: F = {F:.2f}   p-valor = {p_valor:.2e}')
print('Se rechaza H0: hay diferencias significativas.' if p_valor < 0.05 else 'No se rechaza H0.')

In [ ]:
tukey = pairwise_tukeyhsd(endog=resultados_cv['MAE'], groups=resultados_cv['modelo'], alpha=0.05)
print(tukey)

tabla_tukey = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
print('Comparaciones significativas:', int(tabla_tukey['reject'].sum()), 'de', len(tabla_tukey))

In [ ]:
tukey.plot_simultaneous(xlabel='MAE (USD)')
plt.title('Intervalos de confianza de Tukey (95 %) del MAE medio')
plt.show()

mejores = list(resumen.index[:3])
print('MAE medio por modelo:')
print(resumen['MAE'].round(2))
print('\nModelos seleccionados (menor MAE):', mejores)

# Significancia de cada seleccionado frente a cada uno de los no seleccionados
for elegido in mejores:
    filas = tabla_tukey[((tabla_tukey['group1'] == elegido) | (tabla_tukey['group2'] == elegido))]
    otros = filas[~filas['group1'].isin(mejores) | ~filas['group2'].isin(mejores)]
    print(f'{elegido}: significativamente mejor que {int(otros["reject"].sum())} de {len(otros)} modelos no seleccionados')

**Interpretación**

* **ANOVA:** F = 4.39 y p-valor = 0.0009 < 0.05. Se rechaza H₀: no todos los modelos tienen el mismo MAE medio.
* **Tukey:** de las 21 comparaciones, 2 son significativas: **XGBoost vs. KNN** (5.7 USD de diferencia, p = 0.002) y **Random Forest vs. KNN** (5.4 USD, p = 0.004). Las demás diferencias no son significativas: son pequeñas frente a la variación del MAE entre pliegues.

**Selección de los 3 mejores modelos (criterio: menor MAE)**

1. **XGBoost** y 2. **Random Forest**: tienen el menor MAE medio y son los únicos significativamente mejores que otro modelo. Entre ellos no hay diferencia significativa (p = 1.0).
3. **Gradient Boosting**: tercer menor MAE (56.7 USD). Su diferencia con la red neuronal (57.7 USD) no es significativa, así que la prueba no permite separarlos. Se elige porque supera a la red neuronal en MAE, MedAE, MAPE y R² logarítmico, porque está cerca de ser significativamente mejor que KNN (p = 0.069) y porque, con su configuración por defecto (100 árboles de profundidad 3), es el que más puede mejorar con la hiperparametrización.

Los tres modelos seleccionados son métodos de ensamble basados en árboles.

# 8. Hiperparametrización (h)

Cada uno de los 3 modelos seleccionados se optimiza con dos técnicas, **minimizando el MAE** con validación cruzada de 3 pliegues sobre el 70 % de entrenamiento:

1. **GridSearchCV**: prueba todas las combinaciones de una rejilla definida a mano.
2. **Optimización bayesiana (Optuna, algoritmo TPE)**: explora un espacio continuo y usa los intentos anteriores para proponer combinaciones más prometedoras.

Se usan 3 pliegues (en lugar de 10) para que el tiempo sea razonable. Las rejillas y espacios de búsqueda están definidos para los 7 modelos, de modo que el código optimiza los que haya elegido la sección 7.

In [ ]:
kfold3 = KFold(n_splits=3, shuffle=True, random_state=42)
P = 'regressor__modelo__'  # prefijo de los hiperparámetros del modelo dentro del pipeline

rejillas = {
    'Ridge': {P + 'alpha': [0.1, 1, 10, 100]},
    'KNN': {P + 'n_neighbors': [10, 15, 25], P + 'weights': ['uniform', 'distance']},
    'Árbol de decisión': {P + 'max_depth': [8, 12, 16], P + 'min_samples_leaf': [10, 30]},
    'Red neuronal (MLP)': {P + 'hidden_layer_sizes': [(64, 32), (128, 64)], P + 'alpha': [1e-4, 1e-3]},
    'Random Forest': {P + 'max_depth': [None, 20], P + 'min_samples_leaf': [3, 6], P + 'max_features': [0.5]},
    'Gradient Boosting': {P + 'n_estimators': [200, 400], P + 'learning_rate': [0.05, 0.1], P + 'max_depth': [4]},
    'XGBoost': {P + 'n_estimators': [300, 600], P + 'learning_rate': [0.03, 0.06], P + 'max_depth': [6, 8],
                P + 'subsample': [0.8]},
}


def espacio(nombre, trial):
    # Espacio de búsqueda de la optimización bayesiana para cada modelo
    if nombre == 'Ridge':
        return Ridge(alpha=trial.suggest_float('alpha', 1e-3, 100, log=True))
    if nombre == 'KNN':
        return KNeighborsRegressor(n_neighbors=trial.suggest_int('n_neighbors', 5, 50),
                                   weights=trial.suggest_categorical('weights', ['uniform', 'distance']))
    if nombre == 'Árbol de decisión':
        return DecisionTreeRegressor(max_depth=trial.suggest_int('max_depth', 4, 20),
                                     min_samples_leaf=trial.suggest_int('min_samples_leaf', 5, 100), random_state=42)
    if nombre == 'Red neuronal (MLP)':
        capas = trial.suggest_categorical('capas', ['64-32', '128-64', '128-64-32'])
        return MLPRegressor(hidden_layer_sizes=tuple(int(c) for c in capas.split('-')),
                            alpha=trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
                            learning_rate_init=trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
                            early_stopping=True, max_iter=300, random_state=42)
    if nombre == 'Random Forest':
        return RandomForestRegressor(n_estimators=trial.suggest_int('n_estimators', 100, 200),
                                     max_depth=trial.suggest_int('max_depth', 10, 30),
                                     min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
                                     max_features=trial.suggest_float('max_features', 0.3, 0.7),
                                     n_jobs=-1, random_state=42)
    if nombre == 'Gradient Boosting':
        return GradientBoostingRegressor(n_estimators=trial.suggest_int('n_estimators', 100, 400),
                                         learning_rate=trial.suggest_float('learning_rate', 0.02, 0.2, log=True),
                                         max_depth=trial.suggest_int('max_depth', 3, 6),
                                         subsample=trial.suggest_float('subsample', 0.6, 1.0), random_state=42)
    return XGBRegressor(n_estimators=trial.suggest_int('n_estimators', 200, 1000),
                        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                        max_depth=trial.suggest_int('max_depth', 3, 10),
                        subsample=trial.suggest_float('subsample', 0.6, 1.0),
                        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
                        min_child_weight=trial.suggest_int('min_child_weight', 1, 20),
                        n_jobs=-1, random_state=42)


# Número de intentos de Optuna: menos en los modelos más lentos
intentos = {'Gradient Boosting': 12, 'Random Forest': 15}

modelos_ajustados = {}   # pipeline ajustado de cada combinación modelo - técnica
mae_cv = {}              # su MAE en validación cruzada
parametros_finales = {}  # hiperparámetros elegidos

## 8.1 GridSearchCV

In [ ]:
for nombre in mejores:
    inicio = time.time()
    grid = GridSearchCV(construir_pipeline(modelos[nombre]), rejillas[nombre],
                        scoring='neg_mean_absolute_error', cv=kfold3)
    grid.fit(X_train, y_train)
    clave = f'{nombre} - GridSearch'
    modelos_ajustados[clave] = grid.best_estimator_
    mae_cv[clave] = -grid.best_score_
    parametros_finales[clave] = {k.replace(P, ''): v for k, v in grid.best_params_.items()}
    print(f'{clave}: MAE CV = {mae_cv[clave]:.2f} USD | {parametros_finales[clave]} | {time.time() - inicio:.0f} s')

## 8.2 Optimización bayesiana (Optuna)

In [ ]:
for nombre in mejores:
    inicio = time.time()

    def objetivo(trial):
        pipe = construir_pipeline(espacio(nombre, trial))
        return -cross_val_score(pipe, X_train, y_train, cv=kfold3, scoring='neg_mean_absolute_error').mean()

    estudio = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    estudio.optimize(objetivo, n_trials=intentos.get(nombre, 20))

    clave = f'{nombre} - Bayesiana (Optuna)'
    mejor_intento = optuna.trial.FixedTrial(estudio.best_params)
    modelo = construir_pipeline(espacio(nombre, mejor_intento))
    modelo.fit(X_train, y_train)
    modelos_ajustados[clave] = modelo
    mae_cv[clave] = estudio.best_value
    parametros_finales[clave] = estudio.best_params
    print(f'{clave}: MAE CV = {mae_cv[clave]:.2f} USD | {len(estudio.trials)} intentos | {time.time() - inicio:.0f} s')
    print('   ', estudio.best_params)

In [ ]:
tabla_ajuste = pd.DataFrame({'MAE validación cruzada (USD)': mae_cv}).sort_values('MAE validación cruzada (USD)')
tabla_ajuste.round(2)

**Interpretación**

* **La optimización bayesiana superó a GridSearch en los 3 modelos** (XGBoost: 54.91 vs. 55.10 USD; Random Forest: 55.36 vs. 55.48; Gradient Boosting: 55.56 vs. 56.06). GridSearch solo prueba las 4 – 8 combinaciones de la rejilla; Optuna explora valores continuos y concentra los intentos en las zonas prometedoras.
* **Gradient Boosting es el que más mejoró** (de 56.7 a 55.6 USD), porque partía de una configuración por defecto poco profunda. XGBoost y Random Forest ya estaban bien configurados y mejoraron menos. El MAE sin ajustar se midió con 10 pliegues y el ajustado con 3, así que las diferencias pequeñas son orientativas.
* **Mejor modelo: XGBoost optimizado con Optuna** (MAE CV = 54.91 USD): 355 árboles, tasa de aprendizaje 0.030, profundidad 8, `subsample` 0.95, `colsample_bytree` 0.51 y `min_child_weight` 20. Usar la mitad de las variables en cada árbol y exigir un peso mínimo de 20 por hoja regulariza el modelo y reduce el sobreajuste.

# 9. Evaluación con el 30 % de prueba

Los 6 modelos ajustados (3 modelos × 2 técnicas) se evalúan con las mismas 8 medidas sobre el 30 % de prueba. El **mejor modelo** es el de menor MAE en validación cruzada (sección 8); la prueba confirma que el resultado se mantiene con datos nuevos.

In [ ]:
filas = []
for nombre, modelo in modelos_ajustados.items():
    medidas_test = calcular_medidas(y_test, modelo.predict(X_test))
    medidas_test['modelo'] = nombre
    medidas_test['MAE_val_cruzada'] = mae_cv[nombre]
    filas.append(medidas_test)

tabla_test = pd.DataFrame(filas).set_index('modelo').sort_values('MAE_val_cruzada')
tabla_test.round(3)

In [ ]:
mejor = tabla_test.index[0]
modelo_final = modelos_ajustados[mejor]
pred_test = modelo_final.predict(X_test)
print('Mejor modelo:', mejor)
print('Hiperparámetros:', parametros_finales[mejor])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(y_test, pred_test, s=3, alpha=0.3)
ax[0].plot([10, 10000], [10, 10000], 'r--')
ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_xlabel('Precio real (USD, escala log)')
ax[0].set_ylabel('Precio predicho (USD, escala log)')
ax[0].set_title('Real vs. predicho (30 % de prueba)')
ax[1].hist((pred_test - y_test).clip(-400, 400), bins=80)
ax[1].set_xlabel('Error de predicción (USD, recortado a ±400)')
ax[1].set_title('Distribución de los errores')
plt.tight_layout()
plt.show()

error_abs = np.abs(pred_test - y_test)
print(f'Predicciones con error menor de 50 USD: {(error_abs < 50).mean():.1%}')
print(f'Predicciones con error menor de 200 USD: {(error_abs < 200).mean():.1%}')

In [ ]:
# Importancia de las variables del modelo final
modelo_interno = modelo_final.regressor_.named_steps['modelo']
nombres = modelo_final.regressor_.named_steps['preprocesamiento'].get_feature_names_out()

if hasattr(modelo_interno, 'feature_importances_'):
    importancia = pd.Series(modelo_interno.feature_importances_, index=nombres).sort_values(ascending=False)
    importancia.head(15).iloc[::-1].plot.barh(figsize=(8, 6))
    plt.title('15 variables más importantes - modelo final')
    plt.tight_layout()
    plt.show()
    print(importancia.head(10).round(4))
else:
    importancia = pd.Series(dtype=float)
    print('El modelo final no expone importancias de variables.')

## 9.1 Comparación con el modelo base y con la línea base ingenua

Para medir el aporte de la minería de características, se compara el modelo final con:

* **Línea base ingenua:** predice siempre la mediana del precio de entrenamiento.
* **Modelo base de 9 variables:** el ganador de la primera versión del proyecto (XGBoost optimizado con Optuna sobre las 9 variables del notebook de preparación y `dist_centro_km`), entrenado aquí con el mismo 70 % y los mismos hiperparámetros.

In [ ]:
COLS_BASE = ['neighbourhood_group', 'latitude', 'longitude', 'room_type', 'minimum_nights', 'number_of_reviews',
             'reviews_per_month', 'calculated_host_listings_count', 'availability_365']
NUM_BASE = ['latitude', 'longitude', 'dist_centro_km', 'minimum_nights', 'number_of_reviews',
            'reviews_per_month', 'calculated_host_listings_count', 'availability_365']


def preparar_base(df):
    df = df[COLS_BASE].copy()
    df['reviews_per_month'] = df['reviews_per_month'].fillna(0)
    df['minimum_nights'] = df['minimum_nights'].clip(1, 365)
    df['availability_365'] = df['availability_365'].clip(0, 365)
    df['dist_centro_km'] = crear_variables(df.assign(name=''))['dist_times_square_km']
    for col in ['minimum_nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count']:
        df[col] = np.log1p(df[col])
    return df


modelo_base = TransformedTargetRegressor(
    regressor=Pipeline([
        ('preparacion', FunctionTransformer(preparar_base)),
        ('preprocesamiento', ColumnTransformer([
            ('num', StandardScaler(), NUM_BASE),
            ('cat', OneHotEncoder(handle_unknown='ignore'), ['neighbourhood_group', 'room_type'])])),
        ('modelo', XGBRegressor(n_estimators=778, learning_rate=0.015290780725853109, max_depth=9,
                                subsample=0.6595642527846146, colsample_bytree=0.7912892004380444,
                                min_child_weight=1, n_jobs=-1, random_state=42)),
    ]), func=np.log1p, inverse_func=np.expm1)
modelo_base.fit(X_train, y_train)

comparativa = pd.DataFrame({
    'Línea base ingenua (mediana)': calcular_medidas(y_test, np.full(len(y_test), y_train.median())),
    'Modelo base (9 variables)': calcular_medidas(y_test, modelo_base.predict(X_test)),
    f'Modelo final ({mejor})': calcular_medidas(y_test, pred_test),
}).T
comparativa.loc['Línea base ingenua (mediana)', 'R2_log'] = np.nan  # sin variación, el R² log no es informativo
comparativa.round(3)

In [ ]:
base9 = comparativa.loc['Modelo base (9 variables)']
final = comparativa.loc[f'Modelo final ({mejor})']
ingenua = comparativa.loc['Línea base ingenua (mediana)']
for medida in ['MAE', 'MedAE', 'RMSE', 'MAPE']:
    print(f'{medida:<6} reducción frente al modelo base: {(1 - final[medida] / base9[medida]) * 100:5.2f} %   '
          f'| frente a la línea base ingenua: {(1 - final[medida] / ingenua[medida]) * 100:5.2f} %')
print(f'R2_log aumento frente al modelo base: {(final["R2_log"] / base9["R2_log"] - 1) * 100:.2f} %')

**Interpretación de la evaluación con el 30 % de prueba**

* **XGBoost sigue siendo el mejor modelo.** En la prueba, la versión de GridSearch tiene un MAE apenas menor (55.78 frente a 55.91 USD). La diferencia (0.13 USD) es mucho menor que la variación entre pliegues, y la regla de selección se fijó antes con la validación cruzada: elegir con el conjunto de prueba haría que su error dejara de ser una estimación imparcial. Por eso se conserva el modelo de Optuna.
* **No hay sobreajuste:** el MAE de prueba (55.9 USD) es casi igual al de validación cruzada (54.9 USD), y el R² logarítmico también (0.634 frente a 0.621).
* **RMSE (245 USD), MSE (60 076 USD²) y R² en dólares (0.138)** son peores que en validación cruzada porque la prueba contiene alojamientos de 5 000 a 10 000 USD que dominan las medidas cuadráticas. Las medidas robustas (MAE, MedAE, MAPE y R² log) se mantienen estables.
* **Error máximo (9 938 USD):** el alojamiento de 10 000 USD por noche de la prueba, al que el modelo asigna un precio normal. Es un caso aislado: el **75 % de las predicciones tiene un error menor de 50 USD y el 96.5 %, menor de 200 USD**.
* **En la práctica:** para un alojamiento típico el modelo se equivoca unos **23 USD** (MedAE), con un error relativo medio del **30 %** (MAPE). Sirve como precio de referencia para alojamientos comunes, no para el segmento de lujo.
* **Frente a la línea base ingenua** (predecir siempre la mediana), el modelo reduce el MAE un 34 % (de 85.2 a 55.9 USD) y el error mediano a la mitad (de 47.0 a 22.9 USD). Cumple el valor esperado del proyecto (MAE ≤ 64 USD y R² log ≥ 0.50).
* **Frente al modelo base de 9 variables**, la minería de características reduce el MAE un 1.8 % (de 56.9 a 55.9 USD) y el MAPE un 2.0 %, y aumenta el R² logarítmico un 2.2 % (de 0.621 a 0.634). El MedAE y el RMSE casi no cambian (−0.3 % y −0.1 %). La mejora es modesta pero va en la misma dirección en todas las medidas de error: las nuevas variables ayudan sobre todo en los alojamientos que el modelo base estimaba peor.
* **Gráfico real vs. predicho:** los puntos siguen la diagonal, pero el modelo subestima los alojamientos muy caros y sobreestima los muy baratos, lo esperable cuando faltan variables que expliquen los extremos.
* **Importancia de las variables:** el tipo de alojamiento domina (vivienda completa 0.57, habitación privada 0.18, compartida 0.05), seguido de estar en Manhattan (0.06) y del micro-barrio codificado con Target Encoding (0.015). Entre las variables creadas, las más útiles son las palabras de lujo del título (`feat_luxury_count`, 0.010), la disponibilidad todo el año y la distancia a Times Square.

# 10. Pipeline final para despliegue (i)

El modelo final se guarda en un **pipeline** que contiene todas las operaciones para pasar de un registro "crudo" (con el formato de `AB_NYC_2019.csv`) al precio en dólares:

| Paso | Clase | Operación |
|---|---|---|
| 1. Preparación de datos | `FunctionTransformer(preparar_datos)` | Conserva las 11 columnas de entrada (descarta `id`, `host_id`, `host_name`, `last_review`), rellena `reviews_per_month` y `name` vacíos y aplica las reglas de calidad. |
| 2. Ingeniería de características | `FunctionTransformer(crear_variables)` | Distancias a 6 puntos de interés, señales del título, ratios de negocio y `log(1 + x)` de los conteos. |
| 3. Preprocesamiento | `ColumnTransformer` | Estandarización, one-hot de `neighbourhood_group` y `room_type`, Target Encoding de `neighbourhood`. |
| 4. Modelo | modelo optimizado | Ganador de la sección 8. |
| 5. Transformación del objetivo | `TransformedTargetRegressor` | Entrena con `log(1 + price)` y devuelve dólares. |

Para cargarlo se necesita `features.py`. Junto con el modelo se guardan las métricas (`metricas_modelo.json`) y la importancia de las variables, que usa la aplicación.

In [ ]:
modelo_final

In [ ]:
os.makedirs(os.path.join(RAIZ, 'models'), exist_ok=True)
ruta_modelo = os.path.join(RAIZ, 'models', 'modelo_final_airbnb.pkl')
joblib.dump(modelo_final, ruta_modelo)

# Rango de referencia para la app: cuantiles 25 % y 75 % de la razón precio real / precio predicho en la prueba
razon = y_test / pred_test
cuantiles = {'q25': float(razon.quantile(0.25)), 'q75': float(razon.quantile(0.75))}


def a_python(valor):
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating,)):
        return float(valor)
    if isinstance(valor, tuple):
        return list(valor)
    return valor


metricas = {
    'generado_en': pd.Timestamp.now(tz='UTC').isoformat(),
    'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
    'validacion_cruzada_10_pliegues': resumen.round(4).reset_index().to_dict(orient='records'),
    'anova': {'F': float(F), 'p_valor': float(p_valor)},
    'tukey_significativas': int(tabla_tukey['reject'].sum()),
    'modelos_seleccionados': mejores,
    'hiperparametrizacion_mae_cv': {k: float(v) for k, v in mae_cv.items()},
    'mejor_modelo': {'nombre': mejor,
                     'parametros': {k: a_python(v) for k, v in parametros_finales[mejor].items()}},
    'test_modelos_ajustados': tabla_test.round(4).reset_index().to_dict(orient='records'),
    'test_comparativa': comparativa.round(4).reset_index().rename(columns={'index': 'modelo'}).to_dict(orient='records'),
    'rango_referencia_razon': cuantiles,
    'top_caracteristicas': [{'caracteristica': k, 'importancia': float(v)} for k, v in importancia.head(15).items()],
}
with open(os.path.join(RAIZ, 'models', 'metricas_modelo.json'), 'w', encoding='utf-8') as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False, default=a_python)
importancia.rename('importancia').rename_axis('caracteristica').reset_index().to_csv(
    os.path.join(RAIZ, 'models', 'importancia_caracteristicas.csv'), index=False)
resultados_cv.to_csv(os.path.join(RAIZ, 'models', 'resultados_validacion_cruzada.csv'), index=False)

print('Modelo guardado en', ruta_modelo, f'({os.path.getsize(ruta_modelo) / 1e6:.1f} MB)')
print('Rango de referencia (razón real/predicho, 25 % - 75 %):', {k: round(v, 3) for k, v in cuantiles.items()})

## Comprobación del pipeline guardado

Se verifica que el pipeline guardado se carga, da las mismas predicciones que el modelo en memoria y prepara por sí mismo registros nuevos con el formato del archivo original (con identificadores, `reviews_per_month` vacío y valores fuera de las reglas de calidad).

In [ ]:
modelo_cargado = joblib.load(ruta_modelo)
print('¿Predicciones idénticas en el 30 % de prueba?:', np.allclose(modelo_final.predict(X_test), modelo_cargado.predict(X_test)))

nuevos = pd.DataFrame({
    'id': [99990001, 99990002, 99990003],
    'name': ['Luxury loft with skyline view near Times Square', 'Cozy private room near subway', 'Spacious apartment in Astoria'],
    'host_id': [123, 456, 789],
    'host_name': ['Ana', 'Luis', 'Sara'],
    'neighbourhood_group': ['Manhattan', 'Brooklyn', 'Queens'],
    'neighbourhood': ['Midtown', 'Williamsburg', 'Astoria'],
    'latitude': [40.7549, 40.7081, 40.7644],
    'longitude': [-73.9840, -73.9571, -73.9235],
    'room_type': ['Entire home/apt', 'Private room', 'Entire home/apt'],
    'minimum_nights': [2, 3, 1000],              # 1000 está fuera de la regla de calidad
    'number_of_reviews': [25, 0, 4],
    'last_review': ['2019-06-30', None, '2019-05-01'],
    'reviews_per_month': [1.2, None, 0.3],       # vacío: anuncio sin reseñas
    'calculated_host_listings_count': [1, 1, 2],
    'availability_365': [150, 300, 365],
})
nuevos['precio_estimado_USD'] = modelo_cargado.predict(nuevos).round(0)
nuevos[['name', 'neighbourhood', 'room_type', 'precio_estimado_USD']]

El pipeline guardado se carga, reproduce exactamente las predicciones del modelo final y prepara por sí mismo los registros nuevos: descarta las columnas que no usa, rellena las reseñas por mes vacías, ajusta las 1 000 noches mínimas al límite de 365 y calcula todas las variables. Las predicciones son coherentes: el loft de lujo en Midtown es el más caro y la habitación privada en Williamsburg la más económica.

**Resumen del cumplimiento del punto 3**

| Requisito | Cómo se cumple |
|---|---|
| a. Ingeniería de características | 34 características: distancias a 6 puntos de interés, señales del título, ratios de negocio, logaritmos, Target Encoding, one-hot y estandarización |
| b. Balanceo | No aplica (regresión); la asimetría del precio se trata con `log(1 + price)` |
| c. Validación cruzada con el 70 % | 10 pliegues sobre 34 209 registros |
| d. 4 métodos supervisados | Ridge, KNN, árbol de decisión y red neuronal (MLP) |
| e. 3 métodos de ensamble | Random Forest, Gradient Boosting y XGBoost |
| f. Medidas de calidad | 8 medidas (MAE, MedAE, RMSE, MSE, MAPE, error máximo, R² log y R² USD) interpretadas; selección por el menor MAE |
| g. ANOVA y Tukey | F = 4.39, p = 0.0009; seleccionados XGBoost, Random Forest y Gradient Boosting |
| h. GridSearch y optimización bayesiana | 6 modelos ajustados; ganador XGBoost con Optuna (MAE CV = 54.9 USD, MAE prueba = 55.9 USD) |
| i. Pipeline para despliegue | `models/modelo_final_airbnb.pkl`, usado por `app.py` |